# 37-Fold Leave-One-Subject-Out (LOSO) Training

**版本**: v2.0  
**日期**: 2025-01-14  
**策略**: 固定验证集 + 测试集轮换

---

## 流程概览

**固定验证集**: 最后一个subject（按排序）  
**测试集轮换**: 剩余37个subject，每个做一次测试集

```
Fold 01: train=36个, val=subject038, test=subject001
Fold 02: train=36个, val=subject038, test=subject002
Fold 03: train=36个, val=subject038, test=subject003
...
Fold 37: train=36个, val=subject038, test=subject037
```

---

## 设计优势

1. ✅ **验证集固定** → 早停标准一致，便于跨fold比较
2. ✅ **温度缩放公平** → 所有fold在同一验证集上拟合温度
3. ✅ **每个subject（除固定验证集）都做一次测试**
4. ✅ **训练集大小一致** → 每个fold都是36个subject
5. ✅ **符合LOSO理念** → 测试集完全独立

---

## 使用说明

1. 修改下方的 `DATA_ROOT` 指向你的数据目录
2. 调整训练参数（epochs, batch_size等）
3. 运行所有单元格
4. 结果保存在 `runs/loso_37fold/fold_XXX/` 中
5. 汇总结果保存在 `runs/loso_37fold/loso_summary.json`

---

## 1. 配置与导入

In [ ]:
# 配置参数
DATA_ROOT = "/home/jovyan/gpu_space/workspace_jiayi/alex_datasets/downsampling/3d"  # 数据根目录
SEED = 42
EPOCHS = 25  # 完整训练（可根据需要调整，建议先用3测试）
BATCH_SIZE = 256
LR = 1e-5
WEIGHT_DECAY = 1e-5
GRAD_CLIP_NORM = 1.0
USE_CLASS_WEIGHTS = False  # 是否使用类别权重
CLASS_WEIGHT_ALPHA = 0.5
QUALITY_WEIGHTING = False  # 是否使用质量感知样本权重
QUALITY_GAMMA = 1.0
QUALITY_QMIN = 0.2
MAX_VOX_PER_SUBJECT = None  # 显存限制（如需要可设为50000）
BASE_SAVE_DIR = "runs/loso_37fold"  # LOSO结果根目录

print("37-Fold LOSO Training Configuration (Fixed Validation Set):")
print("=" * 70)
print(f"DATA_ROOT: {DATA_ROOT}")
print(f"SEED: {SEED}")
print(f"EPOCHS: {EPOCHS}")
print(f"BATCH_SIZE: {BATCH_SIZE}")
print(f"LR: {LR}")
print(f"WEIGHT_DECAY: {WEIGHT_DECAY}")
print(f"GRAD_CLIP_NORM: {GRAD_CLIP_NORM}")
print(f"USE_CLASS_WEIGHTS: {USE_CLASS_WEIGHTS}")
print(f"QUALITY_WEIGHTING: {QUALITY_WEIGHTING}")
print(f"BASE_SAVE_DIR: {BASE_SAVE_DIR}")
print("=" * 70)

In [ ]:
# 导入依赖
import sys
import json
import numpy as np
from pathlib import Path
from datetime import datetime

# 确保train_runner在路径中
sys.path.insert(0, str(Path.cwd()))

from train_runner import run_single_split

print("✓ 依赖导入成功")

## 2. 验证数据目录并设置固定验证集

In [ ]:
# 检查数据目录
data_root_path = Path(DATA_ROOT)

if not data_root_path.exists():
    raise FileNotFoundError(f"数据根目录不存在: {data_root_path}")

dir_1d = data_root_path / '1d'
dir_3d = data_root_path / '3d'

if not dir_1d.exists():
    raise FileNotFoundError(f"1D数据目录不存在: {dir_1d}")
if not dir_3d.exists():
    raise FileNotFoundError(f"3D数据目录不存在: {dir_3d}")

# 获取所有被试ID（排序）
all_1d_files = sorted(dir_1d.glob('*_1d.npz'))
all_subject_ids = sorted([f.stem.replace('_1d', '') for f in all_1d_files])

print(f"✓ 数据目录验证通过")
print(f"  总被试数: {len(all_subject_ids)}")

# 固定验证集为最后一个subject
FIXED_VAL_SUBJECT = all_subject_ids[-1]
TEST_SUBJECT_POOL = all_subject_ids[:-1]  # 剩余37个用于测试集轮换

print(f"\n37-Fold LOSO 设置:")
print(f"  固定验证集: {FIXED_VAL_SUBJECT}")
print(f"  测试集轮换: {len(TEST_SUBJECT_POOL)} 个subject")
print(f"  训练集大小: 36 个subject (每个fold)")

print(f"\n测试集轮换列表:")
for i, subject_id in enumerate(TEST_SUBJECT_POOL, 1):
    print(f"  Fold {i:2d}: test={subject_id}")

if len(all_subject_ids) != 38:
    print(f"\n⚠️  警告: 期望38个被试，实际找到{len(all_subject_ids)}个")
    print(f"将进行{len(TEST_SUBJECT_POOL)}-fold交叉验证（固定验证集={FIXED_VAL_SUBJECT}）")

## 3. LOSO训练循环

**注意**: 这将运行 37 次完整训练。

**预计时间**: 
- 单个fold: 30-60分钟（取决于EPOCHS和数据规模）
- 37 folds总计: **20-38小时**（EPOCHS=25）

**建议**:
1. 先用 `EPOCHS=3` 运行前2个fold测试流程
2. 确认无误后设置 `EPOCHS=25` 在GPU服务器上overnight运行
3. 可使用分批次运行（见下方fold范围设置）

In [ ]:
# 可选：指定要运行的fold范围（用于分批次运行）
# 例如：START_FOLD=0, END_FOLD=10 只运行前10个fold
START_FOLD = 0  # 开始索引（0-based）
END_FOLD = len(TEST_SUBJECT_POOL)  # 结束索引（不包含），默认运行到最后

print(f"将运行 fold {START_FOLD + 1} 到 fold {END_FOLD} (共{END_FOLD - START_FOLD}个fold)")
print(f"总fold数: {len(TEST_SUBJECT_POOL)}")

In [ ]:
# 创建保存目录
base_save_path = Path(BASE_SAVE_DIR)
base_save_path.mkdir(parents=True, exist_ok=True)

# 初始化结果收集
loso_results = {
    'timestamp': datetime.now().isoformat(),
    'strategy': '37-Fold LOSO with Fixed Validation Set',
    'fixed_val_subject': FIXED_VAL_SUBJECT,
    'n_folds': len(TEST_SUBJECT_POOL),
    'config': {
        'data_root': str(DATA_ROOT),
        'seed': SEED,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'lr': LR,
        'weight_decay': WEIGHT_DECAY,
        'grad_clip_norm': GRAD_CLIP_NORM,
        'use_class_weights': USE_CLASS_WEIGHTS,
        'class_weight_alpha': CLASS_WEIGHT_ALPHA,
        'quality_weighting': QUALITY_WEIGHTING,
        'quality_gamma': QUALITY_GAMMA,
        'quality_qmin': QUALITY_QMIN,
        'max_vox_per_subject': MAX_VOX_PER_SUBJECT
    },
    'folds': []
}

# LOSO循环
for fold_idx in range(START_FOLD, END_FOLD):
    test_subject = TEST_SUBJECT_POOL[fold_idx]
    val_subject = FIXED_VAL_SUBJECT  # 固定验证集
    
    # 训练集 = 除了test和val之外的所有subject
    train_subjects = [s for s in all_subject_ids if s not in [test_subject, val_subject]]
    
    print("\n" + "=" * 80)
    print(f"FOLD {fold_idx + 1}/{len(TEST_SUBJECT_POOL)}")
    print("=" * 80)
    print(f"Test Subject:  {test_subject}")
    print(f"Val Subject:   {val_subject} (FIXED)")
    print(f"Train Subjects: {len(train_subjects)} subjects")
    print("=" * 80)
    
    # 创建fold专用保存目录
    fold_save_dir = base_save_path / f"fold_{fold_idx + 1:02d}_test_{test_subject}"
    
    try:
        # 运行单轮训练
        run_single_split(
            data_root=DATA_ROOT,
            val_id=val_subject,
            test_id=test_subject,
            seed=SEED,  # 所有fold使用相同seed（因为验证集固定）
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            lr=LR,
            weight_decay=WEIGHT_DECAY,
            grad_clip_norm=GRAD_CLIP_NORM,
            use_class_weights=USE_CLASS_WEIGHTS,
            class_weight_alpha=CLASS_WEIGHT_ALPHA,
            quality_weighting=QUALITY_WEIGHTING,
            quality_gamma=QUALITY_GAMMA,
            quality_qmin=QUALITY_QMIN,
            max_vox_per_subject=MAX_VOX_PER_SUBJECT,
            save_dir=str(fold_save_dir)
        )
        
        # 读取本fold的结果
        summary_path = fold_save_dir / 'run_summary.json'
        if summary_path.exists():
            with open(summary_path, 'r') as f:
                fold_summary = json.load(f)
            
            # 收集关键指标
            fold_result = {
                'fold_idx': fold_idx + 1,  # 1-indexed for display
                'test_subject': test_subject,
                'val_subject': val_subject,
                'n_train_subjects': len(train_subjects),
                'status': 'success',
                'best_epoch': fold_summary.get('best_epoch'),
                'test_metrics': {
                    'gross_accuracy': fold_summary['test_metrics']['gross_accuracy'],
                    'nll': fold_summary['test_metrics']['nll'],
                    'brier_score': fold_summary['test_metrics']['brier_score'],
                    'macro_f1': fold_summary['test_metrics']['macro_f1'],
                    'soft_ece': fold_summary['test_metrics'].get('soft_ece'),
                    'soft_ece_after_temp_scaling': fold_summary.get('test_soft_ece_after_temp_scaling'),
                    '3d_soft_dice_macro': fold_summary.get('test_3d_soft_dice_macro')
                },
                'val_metrics': {
                    'gross_accuracy': fold_summary['val_metrics']['gross_accuracy'],
                    'nll': fold_summary['val_metrics']['nll'],
                    'best_val_nll': fold_summary.get('best_val_nll')
                },
                'save_dir': str(fold_save_dir)
            }
        else:
            fold_result = {
                'fold_idx': fold_idx + 1,
                'test_subject': test_subject,
                'val_subject': val_subject,
                'status': 'no_summary',
                'save_dir': str(fold_save_dir)
            }
        
    except Exception as e:
        print(f"\n❌ Fold {fold_idx + 1} failed: {str(e)}")
        import traceback
        traceback.print_exc()
        fold_result = {
            'fold_idx': fold_idx + 1,
            'test_subject': test_subject,
            'val_subject': val_subject,
            'status': 'failed',
            'error': str(e),
            'save_dir': str(fold_save_dir)
        }
    
    loso_results['folds'].append(fold_result)
    
    # 保存中间结果（防止中途中断）
    interim_summary_path = base_save_path / 'loso_summary_interim.json'
    with open(interim_summary_path, 'w') as f:
        json.dump(loso_results, f, indent=2)
    
    print(f"\n✓ Fold {fold_idx + 1} 完成")
    print(f"  中间结果已保存: {interim_summary_path}")

print("\n" + "=" * 80)
print("所有Fold训练完成！")
print("=" * 80)

## 4. 汇总结果

In [ ]:
# 计算汇总统计
successful_folds = [f for f in loso_results['folds'] if f['status'] == 'success']
n_successful = len(successful_folds)
n_total = len(loso_results['folds'])

print(f"成功完成的fold: {n_successful}/{n_total}")

if n_successful > 0:
    # 提取所有成功fold的测试集指标
    metrics_to_aggregate = [
        'gross_accuracy', 'nll', 'brier_score', 'macro_f1', 
        'soft_ece', 'soft_ece_after_temp_scaling', '3d_soft_dice_macro'
    ]
    
    aggregated_metrics = {}
    for metric in metrics_to_aggregate:
        values = []
        for fold in successful_folds:
            val = fold['test_metrics'].get(metric)
            if val is not None:
                values.append(val)
        
        if len(values) > 0:
            aggregated_metrics[metric] = {
                'mean': float(np.mean(values)),
                'std': float(np.std(values)),
                'min': float(np.min(values)),
                'max': float(np.max(values)),
                'median': float(np.median(values)),
                'n_folds': len(values)
            }
    
    loso_results['aggregated_test_metrics'] = aggregated_metrics
    loso_results['n_successful_folds'] = n_successful
    
    # 打印汇总结果
    print("\n" + "=" * 80)
    print(f"LOSO 37-Fold 汇总结果 (固定验证集: {FIXED_VAL_SUBJECT})")
    print("=" * 80)
    for metric, stats in aggregated_metrics.items():
        print(f"\n{metric}:")
        print(f"  Mean ± Std: {stats['mean']:.4f} ± {stats['std']:.4f}")
        print(f"  Median: {stats['median']:.4f}")
        print(f"  Range: [{stats['min']:.4f}, {stats['max']:.4f}]")
        print(f"  N folds: {stats['n_folds']}")
    print("=" * 80)

# 保存最终汇总结果
final_summary_path = base_save_path / 'loso_summary.json'
with open(final_summary_path, 'w') as f:
    json.dump(loso_results, f, indent=2)

print(f"\n✓ 最终汇总结果已保存: {final_summary_path}")

## 5. 可视化结果分布

In [ ]:
import matplotlib.pyplot as plt

if n_successful > 0:
    # 绘制关键指标的boxplot
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f'37-Fold LOSO Results (Fixed Val: {FIXED_VAL_SUBJECT})', 
                 fontsize=16, fontweight='bold')
    
    plot_metrics = [
        ('gross_accuracy', 'Gross Accuracy', axes[0, 0]),
        ('nll', 'Negative Log-Likelihood', axes[0, 1]),
        ('brier_score', 'Brier Score', axes[0, 2]),
        ('macro_f1', 'Macro F1-Score', axes[1, 0]),
        ('soft_ece', 'Soft ECE (Before Temp Scaling)', axes[1, 1]),
        ('3d_soft_dice_macro', '3D Soft Dice (Macro)', axes[1, 2])
    ]
    
    for metric, title, ax in plot_metrics:
        values = []
        for fold in successful_folds:
            val = fold['test_metrics'].get(metric)
            if val is not None:
                values.append(val)
        
        if len(values) > 0:
            # Boxplot
            bp = ax.boxplot([values], vert=True, patch_artist=True)
            bp['boxes'][0].set_facecolor('lightblue')
            bp['boxes'][0].set_alpha(0.7)
            
            # Scatter individual points
            x_pos = np.ones(len(values)) + np.random.normal(0, 0.04, len(values))
            ax.scatter(x_pos, values, alpha=0.5, s=30, color='darkblue')
            
            # Statistics
            mean_val = np.mean(values)
            std_val = np.std(values)
            ax.axhline(mean_val, color='red', linestyle='--', linewidth=2, 
                      label=f'Mean: {mean_val:.4f}')
            
            ax.set_title(title, fontsize=12, fontweight='bold')
            ax.set_ylabel('Value', fontsize=10)
            ax.set_xticks([1])
            ax.set_xticklabels([f'n={len(values)}'])
            ax.legend(fontsize=9)
            ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    
    # 保存图表
    plot_path = base_save_path / 'loso_results_distribution.png'
    plt.savefig(plot_path, dpi=200, bbox_inches='tight')
    print(f"✓ 结果分布图已保存: {plot_path}")
    
    plt.show()
else:
    print("没有成功的fold，无法绘制结果分布")

## 6. 各Fold结果对比表

In [ ]:
import pandas as pd
from IPython.display import display

if n_successful > 0:
    # 创建结果对比表
    table_data = []
    for fold in loso_results['folds']:
        if fold['status'] == 'success':
            row = {
                'Fold': fold['fold_idx'],
                'Test Subject': fold['test_subject'],
                'Best Epoch': fold['best_epoch'],
                'Gross Acc': fold['test_metrics']['gross_accuracy'],
                'NLL': fold['test_metrics']['nll'],
                'Brier': fold['test_metrics']['brier_score'],
                'Macro F1': fold['test_metrics']['macro_f1'],
                'Soft ECE': fold['test_metrics'].get('soft_ece', np.nan),
                'ECE (Calibrated)': fold['test_metrics'].get('soft_ece_after_temp_scaling', np.nan),
                '3D Soft Dice': fold['test_metrics'].get('3d_soft_dice_macro', np.nan)
            }
            table_data.append(row)
    
    df = pd.DataFrame(table_data)
    
    # 添加汇总行
    summary_row = {
        'Fold': 'MEAN',
        'Test Subject': f'All ({n_successful} folds)',
        'Best Epoch': '-',
        'Gross Acc': df['Gross Acc'].mean(),
        'NLL': df['NLL'].mean(),
        'Brier': df['Brier'].mean(),
        'Macro F1': df['Macro F1'].mean(),
        'Soft ECE': df['Soft ECE'].mean(),
        'ECE (Calibrated)': df['ECE (Calibrated)'].mean(),
        '3D Soft Dice': df['3D Soft Dice'].mean()
    }
    std_row = {
        'Fold': 'STD',
        'Test Subject': '-',
        'Best Epoch': '-',
        'Gross Acc': df['Gross Acc'].std(),
        'NLL': df['NLL'].std(),
        'Brier': df['Brier'].std(),
        'Macro F1': df['Macro F1'].std(),
        'Soft ECE': df['Soft ECE'].std(),
        'ECE (Calibrated)': df['ECE (Calibrated)'].std(),
        '3D Soft Dice': df['3D Soft Dice'].std()
    }
    
    df = pd.concat([df, pd.DataFrame([summary_row, std_row])], ignore_index=True)
    
    # 保存为CSV
    csv_path = base_save_path / 'loso_results_table.csv'
    df.to_csv(csv_path, index=False)
    print(f"✓ 结果对比表已保存: {csv_path}")
    
    # 显示表格
    print(f"\n37-Fold LOSO结果对比表 (固定验证集: {FIXED_VAL_SUBJECT}):")
    display(df)
else:
    print("没有成功的fold，无法生成对比表")

## 7. 总结与下一步

### ✅ 训练完成！

你现在有了：

1. **每个fold的完整结果**: `runs/loso_37fold/fold_XX_test_subjectXXX/`
   - 模型权重: `checkpoints/best.pth`
   - 评估指标: `metrics_val.json`, `metrics_test.json`
   - 可视化图表: `figs/` (包括reliability comparison图)
   - 3D预测结果: `pred_3d/`
   - 运行摘要: `run_summary.json`

2. **汇总结果**: `runs/loso_37fold/loso_summary.json`
   - 所有37个fold的详细指标
   - 平均性能和标准差
   - 固定验证集信息

3. **结果可视化**:
   - `loso_results_distribution.png` - 指标分布箱线图
   - `loso_results_table.csv` - 详细对比表（含MEAN和STD）

---

### 📊 论文使用建议

**主要结果表格**:
```
直接使用 loso_results_table.csv 的 MEAN ± STD 行
例如：Gross Acc = 0.8532 ± 0.0123 (37-fold LOSO)
```

**可靠性分析**:
- 每个fold都有 `val_reliability_comparison.png` 和 `test_reliability_comparison.png`
- 展示温度缩放的改善效果（ECE降低）

**Per-class性能**:
- 每个fold的 `metrics_val_3d_advanced.json` 包含 `per_class_ranking`
- 可以分析哪些脑区最难分割（worst_5）

---

### 🔬 可能的后续分析

1. **识别困难样本**:
   - 找出Gross Acc最低的3-5个fold
   - 分析这些被试的特点（数据质量、扫描参数等）

2. **温度缩放效果**:
   - 对比所有fold的 `soft_ece` vs `soft_ece_after_temp_scaling`
   - 量化校准改善的平均幅度

3. **模型集成**:
   - 使用37个fold的模型进行ensemble预测
   - 可能进一步提升性能和校准性

4. **Per-class分析**:
   - 汇总所有fold的per-class Dice
   - 识别全局最难/最易分割的脑区

---

**版本**: v2.0  
**日期**: 2025-01-14  
**策略**: 固定验证集 (最后一个subject) + 37-fold测试集轮换